# Setup python environment

In [17]:
%pip install pandas numpy matplotlib tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Constants

In [29]:
FEATURE_COLS = ["temperature", "pressure", "humidity", "gas_resistance"]
FEATURE_COLS_EXTENDED = FEATURE_COLS # + [f"{c}_diff" for c in FEATURE_COLS]
TARGET_COL = "label"
SEQ_LEN = 10

FILES = [
    "avocado_open_container.csv",
    # "eucalyptus_closed_container_15mins.csv",
    # "lavender_closed_container_15mins.csv",
    "open_room.csv",
    # "room_air_open_forced_15mins.csv",
    "rose_open_container.csv",
    "rose_closed_container.csv",
]

# Load Data

In [43]:
import os
import pandas as pd

def add_fotd_features(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in feature_cols:
        df[col + "_diff"] = df[col].diff()
    df = df.fillna(0)
    return df


def load_dataset(base_dir: str) -> pd.DataFrame:
    dfs = []
    for file in os.listdir(base_dir):
        if file not in FILES: continue
    
        fp = os.path.join(base_dir, file)
        df = pd.read_csv(fp)
        df = df[df['sensor_index'] == 4]  # Only using sensor 1's readings

        # df = add_fotd_features(df, FEATURE_COLS)

        df = df[df['fingerprint_index'] > 0]
        dfs.append(df)

    if len(dfs) == 0:
        raise ValueError(f"No CSVs found under {base_dir}")
    return pd.concat(dfs, ignore_index=True)



data = load_dataset("../data/custom")


print(len(data), " training entries")

6253  training entries


Fix broken fingerprint_index

In [69]:
data[data["label"] == "rose"].sort_values("fingerprint_index").head(20)

,sensor_index,fingerprint_index,position,plate_temperature,heater_duration,temperature,pressure,humidity,gas_resistance,label,fingerprint_index_fixed,label_enc
3206,4,1,0,100,200,21.02,990.48,49.08,0.798599,rose,1,2
3207,4,1,1,150,180,21.04,990.48,49.07,0.788213,rose,1,2
3208,4,1,2,200,160,21.08,990.48,49.01,0.781032,rose,1,2
3209,4,1,3,250,150,21.13,990.48,48.90,0.781032,rose,1,2
3210,4,1,4,300,150,21.17,990.49,48.88,0.781032,rose,1,2
3211,4,1,5,325,140,21.22,990.49,48.80,0.781032,rose,1,2
3212,4,1,6,350,140,21.26,990.50,48.73,0.781032,rose,1,2
3213,4,1,7,380,140,21.30,990.50,48.65,0.781032,rose,1,2
3214,4,1,8,415,140,21.34,990.51,48.58,0.781032,rose,1,2
3215,4,1,9,450,140,21.37,990.52,48.46,0.781032,rose,1,2


# Neural Network

## Data preprocessing

In [55]:
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder


data["gas_resistance"] = np.log1p(data["gas_resistance"])


label_encoder = LabelEncoder()
data['label_enc'] = label_encoder.fit_transform(data[TARGET_COL])


NUM_CLASSES = len(label_encoder.classes_)


scaler = StandardScaler()
data_scaled = data.copy()

data_scaled[FEATURE_COLS_EXTENDED] = scaler.fit_transform(data_scaled[FEATURE_COLS_EXTENDED])

## Create sequences

In [56]:
from collections import defaultdict
import numpy as np

def create_sequences_grouped(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    seq_len: int
):
    X, y = [], []

    # IMPORTANT: group by sensor + label + fingerprint
    grouped = df.groupby(["sensor_index", target_col, "fingerprint_index_fixed"])

    for (sensor, label, fp), group in grouped:
        # sort by position (heater step)
        group = group.sort_values("position")

        data = group[feature_cols].values

        # Only take full cycles
        if len(data) == seq_len:
            X.append(data)
            y.append(label)
        else:
            print(len(data))

    return np.array(X), np.array(y)


X_seq, y_seq = create_sequences_grouped(data_scaled, FEATURE_COLS_EXTENDED, 'label_enc', SEQ_LEN)


print("X_train_seq shape:", X_seq.shape)
print("y_train_seq shape:", y_seq.shape)

4
8
9
9
9
9
9
9
8
9
9
9
9
9
9
9
9
9
9
20
19
13
9
20
19
18
20
18
20
19
20
20
20
20
16
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
8
3
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
7
9
9
9
9
8
9
9
9
9
9
9
9
9
9
9
9
8
9
9
9
9
6
X_train_seq shape: (520, 10, 4)
y_train_seq shape: (520,)


## Build model

In [57]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input

model = Sequential()
model.add(Input(shape=(SEQ_LEN, len(FEATURE_COLS_EXTENDED))))
model.add(LSTM(16, return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(NUM_CLASSES, activation='softmax'))

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 16)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,667 (6.51 KB)

 Trainable params: 1,667 (6.51 KB)

 Non-trainable params: 0 (0.00 B)

## Train model

In [59]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_seq, y_seq,
                    validation_split=0.1,
                    epochs=30,
                    batch_size=64,
                    callbacks=[early_stop],
                    shuffle=True)

Epoch 1/30


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3098 - loss: 1.1777 - val_accuracy: 1.0000 - val_loss: 0.9966
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4038 - loss: 1.1227 - val_accuracy: 1.0000 - val_loss: 1.0106
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4466 - loss: 1.0660 - val_accuracy: 0.9423 - val_loss: 1.0247
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5513 - loss: 0.9677 - val_accuracy: 0.7500 - val_loss: 1.0503
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5406 - loss: 0.9200 - val_accuracy: 0.4615 - val_loss: 1.0668
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6432 - loss: 0.8539 - val_accuracy: 0.4615 - val_loss: 1.0731


## Evaluate model

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

y_pred_probs = model.predict(X_test_seq)
y_pred = np.argmax(y_pred_probs, axis=1)

overall_acc = accuracy_score(y_test_seq, y_pred)
print("Overall Test Accuracy:", overall_acc)

# Confusion matrix for all classes (force full label range)
labels_full = np.arange(NUM_CLASSES)
cm = confusion_matrix(y_test_seq, y_pred, labels=labels_full)

# per-class accuracy (handle zero samples)
denom = cm.sum(axis=1)
per_class_acc = np.divide(cm.diagonal(), denom,
                          out=np.zeros_like(cm.diagonal(), dtype=float),
                          where=denom!=0)

class_names = label_encoder.inverse_transform(labels_full)

for name, acc, n in zip(class_names, per_class_acc, denom):
    print(f"{name:20s} : {acc:.3f}  (n_test={int(n)})")

print("\nClassification Report (only shows classes present in y_test):")
print(classification_report(y_test_seq, y_pred, labels=np.unique(y_test_seq),
                            target_names=label_encoder.inverse_transform(np.unique(y_test_seq))))


In [ ]:
import matplotlib.pyplot as plt

sorted_idx = np.argsort(per_class_acc)[::-1]
sorted_acc = per_class_acc[sorted_idx]
sorted_names = class_names[sorted_idx]

plt.figure(figsize=(14,6))
plt.bar(sorted_names, sorted_acc)
plt.xticks(rotation=90)
plt.ylabel("Accuracy")
plt.xlabel("Smell Class")
plt.title("Per-Class Accuracy")
plt.tight_layout()
plt.show()